In [14]:
from dotenv import load_dotenv
import os
import anthropic

load_dotenv()

# Create client

client = anthropic.Anthropic(api_key=os.environ.get("ANTRHOPIC_API_KEY"))
model = "claude-haiku-4-5"


In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [29]:
import json
def generate_dataset():
  prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
      {
        "task": "Description of task",
      },
      ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """

  messages = []

  add_user_message(messages, prompt)
  add_asst_message(messages, "```json")
  text = chat(messages, stop_sequences=["```"])
  return json.loads(text)

In [30]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that extracts the AWS account ID from an ARN string. The function should take an ARN like 'arn:aws:s3:::my-bucket' or 'arn:aws:ec2:us-east-1:123456789012:instance/i-1234567890abcdef0' and return the 12-digit account ID if present, or None if not found."}, {'task': "Create a JSON object that represents an AWS Lambda environment variables configuration. Include variables for a database connection string, API endpoint URL, and a feature flag boolean, following AWS Lambda's expected format."}, {'task': 'Write a regular expression that matches valid AWS S3 bucket names. The pattern should enforce S3 naming rules: 3-63 characters, lowercase letters, numbers, hyphens and dots, must start and end with a letter or number, and cannot contain consecutive dots or hyphens at the start.'}]


In [31]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)